In [1]:
using MAT               # pour charger .mat
using LinearAlgebra
using Distributions
using Random
using Symbolics

ArgumentError: ArgumentError: Package MAT not found in current path.
- Run `import Pkg; Pkg.add("MAT")` to install the MAT package.

In [2]:
using ModelingToolkit, Symbolics

# États et paramètres
@variables θ1 θ2 ω1 ω2 Tm1 Tm2 N
@variables PG1 PG2 θ1 θ2 PL1 PL2
@variables KL D1 D2 J1 J2 Ks ω0 a1 a2 b1 b2 Pr1 Pr2 P0

x = [θ1, θ2, ω1, ω2, Tm1, Tm2, N]  # États
u = [PG1, PG2, θ1, θ2, PL1, PL2]                       # Entrées

# Sorties
F12 = KL*(θ1 - θ2)
F21 = KL*(θ2 - θ1)
PL1 = F12 - PG1
PL2 = F21 - PG2
Y = [F12, F21, PG1, PG2, PL1, PL2]

# Système dynamique
ωr = (J1*ω1 + J2*ω2)/(J1+J2)

f = [
    ω1 - ω0,
    ω2 - ω0,
    (Tm1 - (PG1/ω1) - D1*(ω1 - ω0))/J1,
    (Tm2 - (PG2/ω2) - D2*(ω2 - ω0))/J2,
    -a1*(Tm1 - (N*Pr1+P0)*ω1) - b1*(ω1 - ω0),
    -a2*(Tm2 - (N*Pr2+P0)*ω2) - b2*(ω2 - ω0),
    Ks*(ωr - ω0)
]

f_linear = [
    ω1 - ω0,
    ω2 - ω0,
    (Tm1 - (PG1/ω0) - D1*(ω1 - ω0))/J1,
    (Tm2 - (PG2/ω0) - D2*(ω2 - ω0))/J2,
    -a1*(Tm1 - (N*Pr1+P0)*ω1) - b1*(ω1 - ω0),
    -a2*(Tm2 - (N*Pr2+P0)*ω2) - b2*(ω2 - ω0),
    Ks*(ωr - ω0)
]




ArgumentError: ArgumentError: Package ModelingToolkit not found in current path.
- Run `import Pkg; Pkg.add("ModelingToolkit")` to install the ModelingToolkit package.

In [3]:
function calcul_matrice_observabilite(f_eqs, y_eqs, states)
    println("1. Calcul des Jacobiennes A et C...")
    # A = df/dx
    A = Symbolics.jacobian(f_eqs, states)
    # C = dY/dx
    C = Symbolics.jacobian(y_eqs, states)
    
    n = length(states)
    
    # Initialisation de la matrice O avec C
    O = C
    
    # Puissance courante de A (A^0, A^1, A^2...)
    A_power = I(n) # Matrice identité initiale (si besoin) ou on commence la boucle direct
    
    println("2. Construction de la matrice O (CA^k)...")
    
    # Terme courant CA^k
    current_term = C
    
    # Boucle pour empiler C, CA, CA^2 ... CA^(n-1)
    # On a déjà C, donc on itère n-1 fois
    for i in 1:(n-1)
        # On calcule le terme suivant : C * A^i
        # Pour optimiser, on peut faire : Terme_precedent * A
        # Cependant, avec Symbolics, multiplier des matrices énormes peut être lourd.
        # On recalcule A^i ou on propage. Propager est plus simple :
        
        current_term = current_term * A
        
        # Concaténation verticale (vcat)
        O = [O; current_term]
    end
    
    return O, A, C
end

calcul_matrice_observabilite (generic function with 1 method)

In [4]:


# --- EXÉCUTION ---

# Calcul
Obs_Matrix, Mat_A, Mat_C = calcul_matrice_observabilite(f_linear, Y, x)

# Affichage des dimensions pour vérifier
println("Dimensions de A : ", size(Mat_A))
println("Dimensions de C : ", size(Mat_C))
println("Dimensions de O : ", size(Obs_Matrix))

# Pour voir un élément spécifique (parce que la matrice entière est illisible en console)
print(Obs_Matrix)

valeurs_numeriques = Dict(
    # --- 1. Paramètres du Réseau & Globaux ---
    KL => 3064.0,       # Raideur pour 2 lignes (MW/rad) [cite: 113]
    Ks => 0.05,         # Gain contrôle secondaire [cite: 99]
    ω0 => 100 * π,      # Fréquence nominale 50Hz (314.159...) [cite: 97]

    # --- 2. Paramètres Générateur 1 (Table 1)  ---
    J1 => 0.4,          # Inertie (Note: valeur très faible, possiblement normalisée)
    D1 => 0.04,         # Amortissement
    Pr1 => 100.0,       # Puissance de régulation (Pr)
    a1 => 100.0,        # Gain alpha (noté "C" dans le tableau)
    b1 => 2000.0,       # Gain beta

    # --- 3. Paramètres Générateur 2 (Table 1)  ---
    J2 => 0.1,          # Inertie
    D2 => 0.02,         # Amortissement
    Pr2 => 50.0,        # Puissance de régulation (Pr)
    a2 => 100.0,        # Gain alpha
    b2 => 2000.0,       # Gain beta

    # --- 4. Point de consigne de Puissance (Attention !) ---
    # Le document a deux consignes distinctes.
    # Ton code utilisait 'P0'. Idéalement, remplace P0 par P1_0 et P2_0 dans tes équations.
    # Je les définis ici :
    P0 => 0.0,          # Valeur par défaut si ton code utilise P0 additivement
    # Si tu modifies ton modèle, utilise ces valeurs pour les consignes :
    # P1_0 => 600.0, 
    # P2_0 => 400.0,

    # --- 5. États Initiaux (Point de Fonctionnement Stable) ---
    
    # Vitesses : Nominales
    ω1 => 100 * π, 
    ω2 => 100 * π,
    
    # Angles : Calculés par l'écoulement de puissance (Load Flow)
    # P_inj1 = P_G1 (600) - P_L1 (400) = +200 MW [cite: 102, 110]
    # F12 = 200 MW. Donc 200 = KL * (θ1 - θ2) => Δθ = 200 / 3064
    θ1 => 200.0 / 3064.0, # approx 0.065 rad
    θ2 => 0.0,            # Angle de référence

    # Couples Mécaniques (Tm)
    # À l'équilibre : Tm = P_Gen / ω0 [cite: 37, 40]
    # P_Gen1 = 600, P_Gen2 = 400
    Tm1 => 600.0 / (100 * π),  # approx 1.91
    Tm2 => 400.0 / (100 * π),  # approx 1.27

    # Variable N (Contrôle secondaire)
    # À l'état stable nominal, N = 0 [cite: 58]
    N => 0.0
)

println("--- Substitution Numérique ---")

# 3. Substitution et conversion en matrice de nombres (Float64)
# On utilise 'substitute' de Symbolics, puis 'Symbolics.value' pour en faire des nombres
O_num = Symbolics.value.(substitute(Obs_Matrix, valeurs_numeriques))
A_num = Symbolics.value.(substitute(Mat_A, valeurs_numeriques))

# 4. Calcul du Rang
r = rank(O_num)
n = length(x) # Devrait être 7
println("Rang calculé : ", r)
println("Nombre d'états : ", n)

UndefVarError: UndefVarError: `f_linear` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [5]:
using ModelingToolkit, Symbolics

# États et paramètres
@variables dθ θ1 θ2 ω1 ω2 Tm1 Tm2 N
@variables PG1 PG2 θ1 θ2 PL1 PL2
@variables KL D1 D2 J1 J2 Ks ω0 a1 a2 b1 b2 Pr1 Pr2 P0

x = [dθ, ω1, ω2, Tm1, Tm2, N]  # États
u = [PG1, PG2, θ1, θ2, PL1, PL2]                       # Entrées

# Sorties
F12 = KL*dθ
F21 = KL*(-dθ)
PL1 = F12 - PG1
PL2 = F21 - PG2
Y = [F12, F21, PG1, PG2, PL1, PL2]

# Système dynamique
ωr = (J1*ω1 + J2*ω2)/(J1+J2)

f = [
    ω1 - ω2, 
    (Tm1 - (PG1/ω1) - D1*(ω1 - ω0))/J1,
    (Tm2 - (PG2/ω2) - D2*(ω2 - ω0))/J2,
    -a1*(Tm1 - (N*Pr1+P0)*ω1) - b1*(ω1 - ω0),
    -a2*(Tm2 - (N*Pr2+P0)*ω2) - b2*(ω2 - ω0),
    Ks*(ωr - ω0)
]

f_linear = [
    ω1 - ω2,
    (Tm1 - (PG1/ω0) - D1*(ω1 - ω0))/J1,
    (Tm2 - (PG2/ω0) - D2*(ω2 - ω0))/J2,
    -a1*(Tm1 - (N*Pr1+P0)*ω1) - b1*(ω1 - ω0),
    -a2*(Tm2 - (N*Pr2+P0)*ω2) - b2*(ω2 - ω0),
    Ks*(ωr - ω0)
]




ArgumentError: ArgumentError: Package ModelingToolkit not found in current path.
- Run `import Pkg; Pkg.add("ModelingToolkit")` to install the ModelingToolkit package.

In [6]:


# --- EXÉCUTION ---

# Calcul
Obs_Matrix, Mat_A, Mat_C = calcul_matrice_observabilite(f_linear, Y, x)

# Affichage des dimensions pour vérifier
println("Dimensions de A : ", size(Mat_A))
println("Dimensions de C : ", size(Mat_C))
println("Dimensions de O : ", size(Obs_Matrix))

# Pour voir un élément spécifique (parce que la matrice entière est illisible en console)
print(Obs_Matrix)

valeurs_numeriques = Dict(
    # --- 1. Paramètres du Réseau & Globaux ---
    KL => 3064.0,       # Raideur pour 2 lignes (MW/rad) [cite: 113]
    Ks => 0.05,         # Gain contrôle secondaire [cite: 99]
    ω0 => 100 * π,      # Fréquence nominale 50Hz (314.159...) [cite: 97]

    # --- 2. Paramètres Générateur 1 (Table 1)  ---
    J1 => 0.4,          # Inertie (Note: valeur très faible, possiblement normalisée)
    D1 => 0.04,         # Amortissement
    Pr1 => 100.0,       # Puissance de régulation (Pr)
    a1 => 100.0,        # Gain alpha (noté "C" dans le tableau)
    b1 => 2000.0,       # Gain beta

    # --- 3. Paramètres Générateur 2 (Table 1)  ---
    J2 => 0.1,          # Inertie
    D2 => 0.02,         # Amortissement
    Pr2 => 50.0,        # Puissance de régulation (Pr)
    a2 => 100.0,        # Gain alpha
    b2 => 2000.0,       # Gain beta

    # --- 4. Point de consigne de Puissance (Attention !) ---
    # Le document a deux consignes distinctes.
    # Ton code utilisait 'P0'. Idéalement, remplace P0 par P1_0 et P2_0 dans tes équations.
    # Je les définis ici :
    P0 => 0.0,          # Valeur par défaut si ton code utilise P0 additivement
    # Si tu modifies ton modèle, utilise ces valeurs pour les consignes :
    # P1_0 => 600.0, 
    # P2_0 => 400.0,

    # --- 5. États Initiaux (Point de Fonctionnement Stable) ---
    
    # Vitesses : Nominales
    ω1 => 100 * π, 
    ω2 => 100 * π,
    
    # Angles : Calculés par l'écoulement de puissance (Load Flow)
    # P_inj1 = P_G1 (600) - P_L1 (400) = +200 MW [cite: 102, 110]
    # F12 = 200 MW. Donc 200 = KL * (θ1 - θ2) => Δθ = 200 / 3064
    θ1 => 200.0 / 3064.0, # approx 0.065 rad
    θ2 => 0.0,            # Angle de référence

    # Couples Mécaniques (Tm)
    # À l'équilibre : Tm = P_Gen / ω0 [cite: 37, 40]
    # P_Gen1 = 600, P_Gen2 = 400
    Tm1 => 600.0 / (100 * π),  # approx 1.91
    Tm2 => 400.0 / (100 * π),  # approx 1.27

    # Variable N (Contrôle secondaire)
    # À l'état stable nominal, N = 0 [cite: 58]
    N => 0.0
)

println("--- Substitution Numérique ---")

# 3. Substitution et conversion en matrice de nombres (Float64)
# On utilise 'substitute' de Symbolics, puis 'Symbolics.value' pour en faire des nombres
O_num = Symbolics.value.(substitute(Obs_Matrix, valeurs_numeriques))
A_num = Symbolics.value.(substitute(Mat_A, valeurs_numeriques))

# 4. Calcul du Rang
r = rank(O_num)
n = length(x) # Devrait être 7
println("Rang calculé : ", r)
println("Nombre d'états : ", n)

UndefVarError: UndefVarError: `f_linear` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [7]:
# # Fonction pour calculer la dérivée de Lie
# function lie_derivative(h::Vector{Num}, f::Vector{Num}, x::Vector{Num})
#     dhdx = [Symbolics.expand_derivatives.(Symbolics.jacobian(h, x))]  # Jacobien
#     Lfh = dhdx[1] * f  # dérivée de Lie
#     return Lfh
# end

# # Calcul de la matrice d'observabilité de Lie
# function lie_observability(f, Y, x, n_deriv)
#     O = Symbolics.zeros(0, length(x))  # matrice vide initiale
#     Lh = Y
#     for k in 0:n_deriv-1
#         # Jacobien pour chaque h
#         for h in Lh
#             J = Symbolics.jacobian([h], x)
#             O = vcat(O, J)
#         end
#         # Calcul de la prochaine dérivée de Lie
#         Lh = lie_derivative(Lh, f, x)
#     end
#     return O
# end

# # Calcul avec n dérivées (ici = taille de x = 7)
# O_Lie = lie_observability(f, Y, x, length(x))
# println("Matrice d'observabilité de Lie :")
# # display(O_Lie)

In [8]:
# using Symbolics, LinearAlgebra

# function evaluate_matrix(O_sym::Matrix{Num}, params::Dict{Num, Float64})
#     O_num = zeros(Float64, size(O_sym)...)
#     for i in 1:size(O_sym,1)
#         for j in 1:size(O_sym,2)
#             # substitution + extraction de la valeur numérique
#             O_num[i,j] = Symbolics.substitute(O_sym[i,j], params) |> Symbolics.value
#         end
#     end
#     return O_num
# end

# # Exemple
# params_num = Dict(
#     θ1=>0.0, θ2=>asin((PL0[1] - P0[1]) / KL), ω1=>2π * 49, ω2=>2π * 49, Tm1=>0.5, Tm2=>0.5, N=>1.0,
#     PG1=>400.0, PG2=>600.0,
#     KL=>3064.0, D1=>0.04, D2=>0.02, J1=>0.4, J2=>0.2, Ks=>0.05, ω0=>2π * 50,
#     a1=>100.0, a2=>100.0, b1=>2000.0, b2=>2000.0, Pr1=>100.0, Pr2=>50.0, P0=>1000.0
# )

# O_num = evaluate_matrix(O_Lie, params_num)
# rank_O = rank(O_num)

# println("Rang numérique de la matrice d'observabilité : ", rank_O)
# println("Dimension du vecteur d'états : ", length(x))

# if rank_O == length(x)
#     println("✅ Le système est localement observable pour ce point d'équilibre.")
# else
#     println("❌ Le système n'est pas observable (rang < dimension de l'état).")
# end
